In [ ]:
import pandas as pd
from pymatgen.core import Composition
from pymatgen.core.periodic_table import Element

In [ ]:
mp_predict = pd.read_csv('./data/predict_materials_project.csv')

In [ ]:
len(mp_predict)

In [ ]:
mp_predict_v2 = mp_predict[mp_predict['avg_score'] > 0.8]

In [ ]:
len(mp_predict_v2)

In [ ]:
score_cols = [col for col in mp_predict.columns if col.startswith("model_") and col.endswith("_score")]
mask = (mp_predict[score_cols] > 0.50).all(axis=1)
mp_predict_v3 = mp_predict_v2[mask]

In [ ]:
len(mp_predict_v3)

In [ ]:
def has_one_tm_and_one_re(formula: str):
    elems = Composition(formula).elements
    re_count = sum(el.is_lanthanoid for el in elems)
    tm_count = sum(el.is_transition_metal and not el.is_lanthanoid for el in elems)
    return (re_count >= 1) and (tm_count >= 1)

mask = mp_predict_v3['formula'].apply(has_one_tm_and_one_re)
mp_predict_v4 = mp_predict_v3[mask]

In [ ]:
def is_rare_earth(el: Element) -> bool:
    return el.is_lanthanoid or el.symbol in ["Sc", "Y"]

def has_one_tm_and_one_re(formula: str) -> bool:
    elems = Composition(formula).elements
    re_count = sum(is_rare_earth(el) for el in elems)
    tm_count = sum(
        el.is_transition_metal and not el.is_lanthanoid and el.symbol not in ["Sc", "Y"]
        for el in elems
    )
    return (re_count >= 1) and (tm_count >= 1)

mask = mp_predict_v3['formula'].apply(has_one_tm_and_one_re)
mp_predict_v4 = mp_predict_v3[mask]

In [ ]:
len(mp_predict_v4)

In [ ]:
theoretical = pd.read_csv('./data/theoretical.csv')

In [ ]:
mp_predict_v5 = mp_predict_v4.merge(theoretical, on='formula', how='left')

In [ ]:
mp_predict_v5 = mp_predict_v5[mp_predict_v5['theoretical'] == False]

In [ ]:
len(mp_predict_v5)

In [ ]:
mp_predict_v5[['formula']].to_csv('./data/predict_pod.csv', index=False)